In [ ]:
import ipywidgets as widgets
from IPython.display import display
from tvbwidgets.ui.connectivity_matrix_editor_widget import ConnectivityMatrixEditor
from tvbwidgets.ui.spacetime_widget import SpaceTimeVisualizerWidget
from tvb.datatypes.connectivity import Connectivity
import numpy as np

# Load and configure the connectivity object
conn = Connectivity.from_file()
conn.configure()


class CustomSpaceTimeEditor(SpaceTimeVisualizerWidget, widgets.VBox):
    

    def __init__(self, connectivity, width=600, height=600):
        SpaceTimeVisualizerWidget.__init__(self, connectivity, width=width, height=height)
        widgets.VBox.__init__(self)
        self.conduction_speed = self.options.children[0].value
        self.children = [widgets.Output()]
        with self.children[0]:
            self.display()

    def on_update_max_tract_length(self, new_max_tract_length):
       
       
        self.plot_overview.clear_output(wait=True)

        self.conduction_speed = self.options.children[0].value

        # Update slider bounds based on new max tract length
        self.options.children[1].max   = new_max_tract_length / self.conduction_speed
        self.options.children[2].max   = new_max_tract_length / self.conduction_speed
        self.options.children[1].min   = self.connectivity.tract_lengths.min() / self.conduction_speed
        self.options.children[2].min   = self.connectivity.tract_lengths.min() / self.conduction_speed
        self.options.children[1].value = self.options.children[1].min
        self.options.children[2].value = self.options.children[2].max

        self.from_time = self.options.children[1].value
        self.to_time   = self.options.children[2].value

       
        self.plot_details.value = self._generate_details()

       
        for idx in range(self.num_slices + 1):
            conn_slice = self._prepare_connectivity(idx)

            
            texture = self.plot.objects[idx]
            texture.attribute   = conn_slice
            texture.color_range = [conn_slice.min(), conn_slice.max()]

            
            colors = self._custom_colormap(conn_slice)
            self.ims[idx].imshow(colors)

        with self.plot_overview:
            display(self.fig)


class CustomMatrixEditor(widgets.VBox, ConnectivityMatrixEditor):
    

    def __init__(self, connectivity, viewer, **kwargs):
        ConnectivityMatrixEditor.__init__(self, connectivity, **kwargs)
        widgets.VBox.__init__(self)
        self.viewer = viewer
        self.children = [widgets.Output()]
        with self.children[0]:
            self.display()

    def on_apply_change(self, change):
       
       
        old_max_tract = self.new_connectivity.tract_lengths.max()

       
        super().on_apply_change(change)

        
        new_max_tract = self.new_connectivity.tract_lengths.max()

        
        if self.viewer is not None and new_max_tract != old_max_tract:
            self.viewer.connectivity = self.new_connectivity
            self.viewer.on_update_max_tract_length(new_max_tract)


viewer = CustomSpaceTimeEditor(connectivity=conn)
editor = CustomMatrixEditor(connectivity=conn, viewer=viewer)

display(viewer)
display(editor)

22-03-2026 04:16:13 - INFO - tvbwidgets - Version: 2.3.2
2026-03-22 16:16:16,284 - WARNING - tvb.basic.readers - File 'hemispheres' not found in ZIP.


CustomSpaceTimeEditor(children=(Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': "HBox(c…

CustomMatrixEditor(children=(Output(),))